# Spatial POD + Graph Encoder + Temporal Heads (LSTM / Transformer / Mamba)

Production-grade notebook for **large-scale CFD spatiotemporal forecasting** on ~2.1M mesh cells.

**Pipeline overview (shared spatial stage, swapped temporal head):**
1. **POD on physical variables only** (`pressure`, `u`, `v`) using training window
2. **Sparse graph** from (`x`, `y`) with kNN or radius (no dense NxN adjacency)
3. **Shared graph encoder** (GraphSAGE / GCN) with projection to POD coefficients
4. **Temporal head** in POD space (LSTM / Transformer / Mamba)
5. **Reconstruction** to physical space + metrics

Artifacts are cached under:
- `outputs/spatial_pod_gcn/`
- `checkpoints/spatial_pod_gcn/`

## 1 — Config

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import json
import os


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here] + list(here.parents):
        if (parent / ".git").exists():
            return parent
    return here


REPO_ROOT = find_repo_root()


@dataclass
class Config:
    # Profiles
    RUN_PROFILE: str = "debug"  # debug | kaggle_balanced | kaggle_full

    # Data
    USE_MOCK_DATA: bool = True
    DATA_ROOT: Path = Path("data")
    PRESSURE_CSV: Optional[Path] = None
    U_VELOCITY_CSV: Optional[Path] = None
    V_VELOCITY_CSV: Optional[Path] = None
    MAX_CELLS: Optional[int] = None
    MAX_TIMESTEPS: Optional[int] = None
    CSV_CHUNK_ROWS: int = 50_000

    # Training splits (time-axis only)
    TRAIN_RATIO: float = 0.7
    VAL_RATIO: float = 0.15

    # Temporal windows
    INPUT_WINDOW: int = 20
    FORECAST_HORIZON: int = 10

    # POD
    POD_MODES: Optional[int] = 64
    POD_EXPLAINED_VAR: Optional[float] = None  # e.g., 0.99
    POD_MAX_MODES: int = 128
    POD_BATCH_TIME: int = 8
    POD_STORE_DTYPE: str = "float32"

    # Graph
    GRAPH_MODE: str = "knn"  # knn | radius
    K_NEIGHBORS: int = 12
    RADIUS: Optional[float] = None
    GRAPH_CHUNK_SIZE: int = 50_000
    GRAPH_LOADER: str = "cluster"  # neighbor | cluster
    CLUSTER_PARTS: int = 1500
    CLUSTER_BATCH_SIZE: int = 1

    # Encoder
    ENCODER_TYPE: str = "graphsage"  # graphsage | gcn
    ENCODER_HIDDEN: int = 64
    ENCODER_LAYERS: int = 2
    ENCODER_DROPOUT: float = 0.1
    USE_NODE_COORDS: bool = True
    USE_NODE_FIELDS: bool = True

    # Coefficient projection
    PROJ_HIDDEN: int = 128
    PROJ_DROPOUT: float = 0.1

    # Temporal heads
    TEMPORAL_HIDDEN: int = 128
    TEMPORAL_LAYERS: int = 2
    TEMPORAL_DROPOUT: float = 0.1
    TRANSFORMER_HEADS: int = 4
    TRANSFORMER_FF: int = 256

    # Training
    USE_JOINT_TRAINING: bool = True
    USE_DIRECT_POD_COEFFS: bool = False  # If True, skip spatial encoder regression
    TRAIN_SPATIAL_ENCODER: bool = True
    SPATIAL_TIME_STRIDE: int = 1
    EPOCHS: int = 30
    SPATIAL_EPOCHS: int = 8
    BATCH_SIZE: int = 16
    LR: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    PATIENCE: int = 6
    GRAD_CLIP: float = 1.0
    GRAD_ACCUM_STEPS: int = 1
    USE_AMP: bool = True
    RESUME: bool = True

    # Output
    OUTPUT_DIR: Path = Path("outputs/spatial_pod_gcn")
    CHECKPOINT_DIR: Path = Path("checkpoints/spatial_pod_gcn")
    CACHE_DIR: Path = Path("outputs/spatial_pod_gcn/cache")

    # Visualization
    MAP_TIMESTEPS: Tuple[int, ...] = (0, 2, 5)
    TIMESERIES_CELLS: int = 4
    PLOT_MAX_CELLS: int = 120_000

    # Reproducibility
    SEED: int = 2027


PROFILE_PRESETS = {
    "debug": {
        "MAX_CELLS": 2000,
        "MAX_TIMESTEPS": 120,
        "EPOCHS": 4,
        "SPATIAL_EPOCHS": 2,
        "SPATIAL_TIME_STRIDE": 4,
        "BATCH_SIZE": 8,
        "INPUT_WINDOW": 12,
        "FORECAST_HORIZON": 6,
        "POD_MODES": 16,
    },
    "kaggle_balanced": {
        "MAX_CELLS": 120_000,
        "MAX_TIMESTEPS": 400,
        "EPOCHS": 15,
        "SPATIAL_EPOCHS": 4,
        "SPATIAL_TIME_STRIDE": 2,
        "BATCH_SIZE": 8,
        "INPUT_WINDOW": 20,
        "FORECAST_HORIZON": 10,
        "POD_MODES": 48,
    },
    "kaggle_full": {
        "MAX_CELLS": None,
        "MAX_TIMESTEPS": None,
        "EPOCHS": 30,
        "SPATIAL_EPOCHS": 8,
        "SPATIAL_TIME_STRIDE": 1,
        "BATCH_SIZE": 4,
        "INPUT_WINDOW": 30,
        "FORECAST_HORIZON": 20,
        "POD_MODES": 64,
    },
}


def apply_profile(cfg: Config) -> Config:
    preset = PROFILE_PRESETS.get(cfg.RUN_PROFILE, {})
    for key, value in preset.items():
        setattr(cfg, key, value)
    if cfg.POD_EXPLAINED_VAR is not None:
        cfg.POD_MODES = None
    if not cfg.USE_DIRECT_POD_COEFFS:
        cfg.TRAIN_SPATIAL_ENCODER = True
    return cfg


cfg = apply_profile(Config())

OUTPUT_DIR = REPO_ROOT / cfg.OUTPUT_DIR
CHECKPOINT_DIR = REPO_ROOT / cfg.CHECKPOINT_DIR
CACHE_DIR = REPO_ROOT / cfg.CACHE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Run profile:", cfg.RUN_PROFILE)
print("Output dir:", OUTPUT_DIR)

## 2 — Imports & environment checks

In [ ]:
import gc
import math
import random
import time
import warnings
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from sklearn.decomposition import IncrementalPCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings("ignore")

try:
    from torch_geometric.data import Data
    from torch_geometric.loader import NeighborLoader, ClusterLoader, ClusterData
    from torch_geometric.nn import SAGEConv, GCNConv, global_mean_pool
    HAS_PYG = True
    PYG_ERROR = None
except Exception as exc:
    HAS_PYG = False
    PYG_ERROR = str(exc)

try:
    from mamba_ssm import Mamba
    HAS_MAMBA = True
    MAMBA_ERROR = None
except Exception as exc:
    HAS_MAMBA = False
    MAMBA_ERROR = str(exc)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyG available: {HAS_PYG}")
if not HAS_PYG:
    print("  PyG import error:", PYG_ERROR)
print(f"Mamba available: {HAS_MAMBA}")
if not HAS_MAMBA:
    print("  Mamba import error:", MAMBA_ERROR)

## 3 — Data loading

In [ ]:
from typing import Any


def generate_mock_data(n_cells: int, n_timesteps: int) -> Tuple[Dict[str, np.ndarray], np.ndarray]:
    rng = np.random.default_rng(cfg.SEED)
    coords = rng.uniform(0.0, 1.0, size=(n_cells, 2)).astype(np.float32)
    t = np.linspace(0.0, 4.0 * math.pi, n_timesteps, dtype=np.float32)

    pressure = (np.sin(t)[None, :] + 0.1 * rng.standard_normal((n_cells, n_timesteps)))
    u_vel = (np.cos(t)[None, :] + 0.1 * rng.standard_normal((n_cells, n_timesteps)))
    v_vel = (np.sin(0.5 * t)[None, :] + 0.1 * rng.standard_normal((n_cells, n_timesteps)))

    fields = {
        "pressure": pressure.astype(np.float32),
        "u": u_vel.astype(np.float32),
        "v": v_vel.astype(np.float32),
    }
    return fields, coords


def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8") as f:
        return max(0, sum(1 for _ in f) - 1)


def infer_csv_timesteps(path: Path) -> int:
    header = pd.read_csv(path, nrows=0)
    return max(0, len(header.columns) - 2)


def load_csv_to_memmap(
    path: Path,
    out_path: Path,
    coords_out_path: Optional[Path] = None,
    max_cells: Optional[int] = None,
    max_timesteps: Optional[int] = None,
    chunk_rows: int = 50_000,
    include_coords: bool = True,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    if out_path.exists():
        data = np.load(out_path, mmap_mode="r+")
        coords = np.load(coords_out_path, mmap_mode="r+") if coords_out_path and coords_out_path.exists() else None
        return data, coords

    n_rows = count_csv_rows(path)
    if max_cells is not None:
        n_rows = min(n_rows, max_cells)

    n_timesteps = infer_csv_timesteps(path)
    if max_timesteps is not None:
        n_timesteps = min(n_timesteps, max_timesteps)

    if include_coords:
        usecols = list(range(0, 2 + n_timesteps))
    else:
        usecols = list(range(2, 2 + n_timesteps))

    data = np.lib.format.open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(n_rows, n_timesteps),
    )

    coords = None
    if include_coords and coords_out_path is not None:
        coords = np.lib.format.open_memmap(
            coords_out_path,
            mode="w+",
            dtype=np.float32,
            shape=(n_rows, 2),
        )

    row_start = 0
    for chunk in pd.read_csv(path, chunksize=chunk_rows, usecols=usecols):
        if row_start >= n_rows:
            break
        if row_start + len(chunk) > n_rows:
            chunk = chunk.iloc[: n_rows - row_start]
        if include_coords and coords is not None:
            coords[row_start : row_start + len(chunk)] = chunk.iloc[:, :2].values.astype(np.float32)
            values = chunk.iloc[:, 2:].values.astype(np.float32)
        else:
            values = chunk.values.astype(np.float32)
        data[row_start : row_start + len(chunk)] = values
        row_start += len(chunk)

    return data, coords


def resolve_data_paths(cfg: Config) -> Dict[str, Path]:
    data_root = REPO_ROOT / cfg.DATA_ROOT
    data_root.mkdir(parents=True, exist_ok=True)
    return {
        "pressure": (cfg.PRESSURE_CSV or data_root / "pressure.csv"),
        "u": (cfg.U_VELOCITY_CSV or data_root / "u_velocity.csv"),
        "v": (cfg.V_VELOCITY_CSV or data_root / "v_velocity.csv"),
    }


def load_fields(cfg: Config) -> Tuple[Dict[str, np.ndarray], np.ndarray]:
    if cfg.USE_MOCK_DATA:
        print("Using synthetic mock data.")
        return generate_mock_data(cfg.MAX_CELLS or 2000, cfg.MAX_TIMESTEPS or 120)

    paths = resolve_data_paths(cfg)
    missing = [name for name, path in paths.items() if not Path(path).exists()]
    if missing:
        print("Missing CSVs:", missing, "-> falling back to mock data.")
        return generate_mock_data(cfg.MAX_CELLS or 2000, cfg.MAX_TIMESTEPS or 120)

    n_timesteps = min(infer_csv_timesteps(paths["pressure"]), infer_csv_timesteps(paths["u"]), infer_csv_timesteps(paths["v"]))
    if cfg.MAX_TIMESTEPS is not None:
        n_timesteps = min(n_timesteps, cfg.MAX_TIMESTEPS)

    coords_path = CACHE_DIR / "coords.npy"
    pressure_path = CACHE_DIR / "pressure.npy"
    u_path = CACHE_DIR / "u.npy"
    v_path = CACHE_DIR / "v.npy"

    print("Loading pressure CSV -> memmap")
    pressure, coords = load_csv_to_memmap(
        paths["pressure"],
        pressure_path,
        coords_out_path=coords_path,
        max_cells=cfg.MAX_CELLS,
        max_timesteps=n_timesteps,
        chunk_rows=cfg.CSV_CHUNK_ROWS,
        include_coords=True,
    )
    print("Loading u-velocity CSV -> memmap")
    u, _ = load_csv_to_memmap(
        paths["u"],
        u_path,
        coords_out_path=None,
        max_cells=cfg.MAX_CELLS,
        max_timesteps=n_timesteps,
        chunk_rows=cfg.CSV_CHUNK_ROWS,
        include_coords=False,
    )
    print("Loading v-velocity CSV -> memmap")
    v, _ = load_csv_to_memmap(
        paths["v"],
        v_path,
        coords_out_path=None,
        max_cells=cfg.MAX_CELLS,
        max_timesteps=n_timesteps,
        chunk_rows=cfg.CSV_CHUNK_ROWS,
        include_coords=False,
    )

    fields = {
        "pressure": pressure,
        "u": u,
        "v": v,
    }
    return fields, coords


fields_raw, coords = load_fields(cfg)

n_cells = coords.shape[0]
n_timesteps = next(iter(fields_raw.values())).shape[1]
print(f"Loaded cells={n_cells:,}, timesteps={n_timesteps}")

## 4 — Normalization (train-only fit)

In [ ]:
def split_time_indices(n_timesteps: int, train_ratio: float, val_ratio: float) -> Tuple[int, int]:
    train_end = max(1, int(n_timesteps * train_ratio))
    val_end = max(train_end + 1, int(n_timesteps * (train_ratio + val_ratio)))
    val_end = min(val_end, n_timesteps - 1)
    return train_end, val_end


def streaming_mean_std(data: np.ndarray, time_slice: slice, chunk_time: int = 8) -> Tuple[float, float]:
    t_start, t_stop, _ = time_slice.indices(data.shape[1])
    total = 0
    sum_vals = 0.0
    sum_sq = 0.0
    for t0 in range(t_start, t_stop, chunk_time):
        t1 = min(t_stop, t0 + chunk_time)
        chunk = np.asarray(data[:, t0:t1], dtype=np.float64)
        total += chunk.size
        sum_vals += chunk.sum()
        sum_sq += (chunk ** 2).sum()
    mean = sum_vals / max(total, 1)
    var = max(sum_sq / max(total, 1) - mean ** 2, 1e-12)
    std = math.sqrt(var)
    return float(mean), float(std)


def normalize_fields(
    fields: Dict[str, np.ndarray],
    train_slice: slice,
    cache_dir: Path,
    chunk_time: int = 8,
) -> Tuple[Dict[str, np.ndarray], Dict[str, Dict[str, float]]]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    stats = {}
    normalized = {}

    for name, data in fields.items():
        norm_path = cache_dir / f"{name}_norm.npy"
        stats_path = cache_dir / f"{name}_norm.json"
        if norm_path.exists() and stats_path.exists():
            normalized[name] = np.load(norm_path, mmap_mode="r+")
            stats[name] = json.loads(stats_path.read_text())
            continue

        mean, std = streaming_mean_std(data, train_slice, chunk_time=chunk_time)
        stats[name] = {"mean": mean, "std": std}
        stats_path.write_text(json.dumps(stats[name], indent=2))

        norm_memmap = np.lib.format.open_memmap(
            norm_path,
            mode="w+",
            dtype=np.float32,
            shape=data.shape,
        )

        for t0 in range(0, data.shape[1], chunk_time):
            t1 = min(data.shape[1], t0 + chunk_time)
            chunk = np.asarray(data[:, t0:t1], dtype=np.float32)
            norm_memmap[:, t0:t1] = (chunk - mean) / std
        normalized[name] = norm_memmap

    return normalized, stats


def normalize_coords(coords: np.ndarray, cache_dir: Path) -> Tuple[np.ndarray, Dict[str, float]]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    coords_path = cache_dir / "coords_norm.npy"
    stats_path = cache_dir / "coords_norm.json"
    if coords_path.exists() and stats_path.exists():
        return np.load(coords_path, mmap_mode="r+"), json.loads(stats_path.read_text())

    mean = coords.mean(axis=0)
    std = coords.std(axis=0) + 1e-8
    coords_norm = ((coords - mean) / std).astype(np.float32)

    np.save(coords_path, coords_norm)
    stats = {
        "mean_x": float(mean[0]),
        "mean_y": float(mean[1]),
        "std_x": float(std[0]),
        "std_y": float(std[1]),
    }
    stats_path.write_text(json.dumps(stats, indent=2))
    return coords_norm, stats


train_end, val_end = split_time_indices(n_timesteps, cfg.TRAIN_RATIO, cfg.VAL_RATIO)
train_slice = slice(0, train_end)
val_slice = slice(train_end, val_end)
test_slice = slice(val_end, n_timesteps)

norm_fields, norm_stats = normalize_fields(fields_raw, train_slice, CACHE_DIR, chunk_time=cfg.POD_BATCH_TIME)
coords_norm, coords_stats = normalize_coords(coords, CACHE_DIR)

split_path = CACHE_DIR / "time_splits.json"
if not split_path.exists():
    split_path.write_text(
        json.dumps({"train_end": train_end, "val_end": val_end, "n_timesteps": n_timesteps}, indent=2)
    )

print(f"Time splits -> train_end={train_end}, val_end={val_end}, test_end={n_timesteps}")

## 5 — POD fitting/loading (physical variables only)

In [ ]:
from typing import NamedTuple


class PODArtifacts(NamedTuple):
    basis: np.ndarray
    mean: np.ndarray
    explained: np.ndarray
    coeffs: np.ndarray


def fit_pod_incremental(
    data: np.ndarray,
    train_slice: slice,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    t_start, t_stop, _ = train_slice.indices(data.shape[1])
    n_samples = t_stop - t_start
    n_features = data.shape[0]

    max_modes = cfg.POD_MODES or cfg.POD_MAX_MODES
    max_modes = min(max_modes, n_samples, n_features)
    ipca = IncrementalPCA(n_components=max_modes, batch_size=cfg.POD_BATCH_TIME)

    for t0 in range(t_start, t_stop, cfg.POD_BATCH_TIME):
        t1 = min(t_stop, t0 + cfg.POD_BATCH_TIME)
        batch = np.asarray(data[:, t0:t1], dtype=np.float32).T
        ipca.partial_fit(batch)

    components = ipca.components_.astype(np.float32)
    explained = ipca.explained_variance_ratio_.astype(np.float32)
    if cfg.POD_EXPLAINED_VAR is not None:
        cumulative = np.cumsum(explained)
        k = int(np.searchsorted(cumulative, cfg.POD_EXPLAINED_VAR) + 1)
        components = components[:k]
        explained = explained[:k]
    mean = ipca.mean_.astype(np.float32)
    return components, mean, explained


def compute_pod_coeffs(
    data: np.ndarray,
    basis: np.ndarray,
    mean: np.ndarray,
    out_path: Path,
    batch_time: int,
) -> np.ndarray:
    n_cells, n_timesteps = data.shape
    coeffs = np.lib.format.open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(n_timesteps, basis.shape[0]),
    )
    for t0 in range(0, n_timesteps, batch_time):
        t1 = min(n_timesteps, t0 + batch_time)
        batch = np.asarray(data[:, t0:t1], dtype=np.float32).T
        centered = batch - mean
        coeffs[t0:t1] = centered @ basis.T
    return coeffs


def load_or_fit_pod(name: str, data: np.ndarray, cfg: Config) -> PODArtifacts:
    basis_path = CACHE_DIR / f"pod_{name}_basis.npy"
    mean_path = CACHE_DIR / f"pod_{name}_mean.npy"
    explained_path = CACHE_DIR / f"pod_{name}_explained.npy"
    coeffs_path = CACHE_DIR / f"pod_{name}_coeffs.npy"

    if basis_path.exists() and mean_path.exists() and explained_path.exists() and coeffs_path.exists():
        return PODArtifacts(
            basis=np.load(basis_path, mmap_mode="r+"),
            mean=np.load(mean_path, mmap_mode="r+"),
            explained=np.load(explained_path, mmap_mode="r+"),
            coeffs=np.load(coeffs_path, mmap_mode="r+"),
        )

    print(f"Fitting POD for {name}...")
    basis, mean, explained = fit_pod_incremental(data, train_slice, cfg)

    np.save(basis_path, basis.astype(cfg.POD_STORE_DTYPE))
    np.save(mean_path, mean.astype(cfg.POD_STORE_DTYPE))
    np.save(explained_path, explained.astype(np.float32))

    coeffs = compute_pod_coeffs(data, basis, mean, coeffs_path, batch_time=cfg.POD_BATCH_TIME)

    return PODArtifacts(basis=basis, mean=mean, explained=explained, coeffs=coeffs)


pod_artifacts = {
    name: load_or_fit_pod(name, norm_fields[name], cfg)
    for name in ("pressure", "u", "v")
}

for name, art in pod_artifacts.items():
    print(
        f"{name}: basis={art.basis.shape}, coeffs={art.coeffs.shape}, explained={art.explained.sum():.4f}"
    )

## 6 — Graph construction/loading (sparse)

In [ ]:
def build_knn_graph(coords: np.ndarray, k: int, chunk_size: int) -> Tuple[np.ndarray, np.ndarray]:
    tree = cKDTree(coords)
    n_cells = coords.shape[0]
    edge_src = []
    edge_dst = []
    edge_w = []

    for start in range(0, n_cells, chunk_size):
        end = min(n_cells, start + chunk_size)
        dist, idx = tree.query(coords[start:end], k=k + 1)
        src = np.repeat(np.arange(start, end), k)
        dst = idx[:, 1:].reshape(-1)
        w = 1.0 / (dist[:, 1:].reshape(-1) + 1e-6)
        edge_src.append(src)
        edge_dst.append(dst)
        edge_w.append(w)

    edge_index = np.vstack([np.concatenate(edge_src), np.concatenate(edge_dst)]).astype(np.int64)
    edge_weight = np.concatenate(edge_w).astype(np.float32)
    return edge_index, edge_weight


def build_radius_graph(coords: np.ndarray, radius: float, max_neighbors: int, chunk_size: int) -> Tuple[np.ndarray, np.ndarray]:
    tree = cKDTree(coords)
    n_cells = coords.shape[0]
    edge_src = []
    edge_dst = []
    edge_w = []

    for start in range(0, n_cells, chunk_size):
        end = min(n_cells, start + chunk_size)
        chunk_coords = coords[start:end]
        neighbors = tree.query_ball_point(chunk_coords, r=radius)
        for i, neigh in enumerate(neighbors):
            if start + i in neigh:
                neigh.remove(start + i)
            if max_neighbors is not None and len(neigh) > max_neighbors:
                neigh = neigh[:max_neighbors]
            src = np.full(len(neigh), start + i, dtype=np.int64)
            dst = np.array(neigh, dtype=np.int64)
            if len(dst) == 0:
                continue
            dists = np.linalg.norm(coords[dst] - coords[start + i], axis=1)
            w = 1.0 / (dists + 1e-6)
            edge_src.append(src)
            edge_dst.append(dst)
            edge_w.append(w)

    edge_index = np.vstack([np.concatenate(edge_src), np.concatenate(edge_dst)]).astype(np.int64)
    edge_weight = np.concatenate(edge_w).astype(np.float32)
    return edge_index, edge_weight


def load_or_build_graph(cfg: Config, coords: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    edge_path = CACHE_DIR / "graph_edges.npz"
    if edge_path.exists():
        data = np.load(edge_path)
        return data["edge_index"], data["edge_weight"]

    if cfg.GRAPH_MODE == "radius":
        if cfg.RADIUS is None:
            raise ValueError("RADIUS must be set when GRAPH_MODE='radius'")
        edge_index, edge_weight = build_radius_graph(
            coords, cfg.RADIUS, cfg.K_NEIGHBORS, cfg.GRAPH_CHUNK_SIZE
        )
    else:
        edge_index, edge_weight = build_knn_graph(coords, cfg.K_NEIGHBORS, cfg.GRAPH_CHUNK_SIZE)

    np.savez_compressed(edge_path, edge_index=edge_index, edge_weight=edge_weight)
    return edge_index, edge_weight


edge_index, edge_weight = load_or_build_graph(cfg, coords_norm)
print(f"Graph edges: {edge_index.shape[1]:,} (k={cfg.K_NEIGHBORS})")

## 7 — Dataset & loaders (POD coefficients)

In [ ]:
class CoeffSequenceDataset(Dataset):
    def __init__(
        self,
        input_coeffs: np.ndarray,
        target_coeffs: np.ndarray,
        input_window: int,
        horizon: int,
        start_idx: int,
        end_idx: int,
    ) -> None:
        self.input_coeffs = input_coeffs
        self.target_coeffs = target_coeffs
        self.input_window = input_window
        self.horizon = horizon
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.length = max(0, end_idx - start_idx - input_window - horizon + 1)
        self.max_start = start_idx + self.length - 1

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        t0 = self.start_idx + idx
        x = self.input_coeffs[t0 : t0 + self.input_window]
        y = self.target_coeffs[t0 + self.input_window : t0 + self.input_window + self.horizon]
        return torch.from_numpy(np.asarray(x, dtype=np.float32)), torch.from_numpy(np.asarray(y, dtype=np.float32))


def build_coeff_matrix(pod_artifacts: Dict[str, PODArtifacts], use_joint: bool) -> Tuple[Optional[np.ndarray], Dict[str, slice]]:
    slices = {}
    coeff_list = []
    start = 0
    for name in ("pressure", "u", "v"):
        coeffs = np.asarray(pod_artifacts[name].coeffs, dtype=np.float32)
        if use_joint:
            coeff_list.append(coeffs)
            end = start + coeffs.shape[1]
            slices[name] = slice(start, end)
            start = end
        else:
            slices[name] = slice(0, coeffs.shape[1])
    if use_joint:
        coeff_matrix = np.concatenate(coeff_list, axis=1)
    else:
        coeff_matrix = None
    return coeff_matrix, slices


def build_dataloaders(
    input_coeffs: np.ndarray,
    target_coeffs: np.ndarray,
    cfg: Config,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_ds = CoeffSequenceDataset(input_coeffs, target_coeffs, cfg.INPUT_WINDOW, cfg.FORECAST_HORIZON, 0, train_end)
    val_ds = CoeffSequenceDataset(input_coeffs, target_coeffs, cfg.INPUT_WINDOW, cfg.FORECAST_HORIZON, train_end, val_end)
    test_ds = CoeffSequenceDataset(input_coeffs, target_coeffs, cfg.INPUT_WINDOW, cfg.FORECAST_HORIZON, val_end, n_timesteps)

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, test_loader


coeff_matrix_joint, coeff_slices = build_coeff_matrix(pod_artifacts, cfg.USE_JOINT_TRAINING)
if cfg.USE_JOINT_TRAINING:
    train_loader, val_loader, test_loader = build_dataloaders(coeff_matrix_joint, coeff_matrix_joint, cfg)
    print("Joint coeff matrix:", coeff_matrix_joint.shape)
else:
    print("Per-variable training enabled.")

## 8 — Model definitions (graph encoder + temporal heads)

In [ ]:
class SimpleSAGEConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.lin = nn.Linear(in_channels * 2, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        row, col = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, row, x[col])
        deg = torch.bincount(row, minlength=x.size(0)).clamp(min=1).unsqueeze(1)
        agg = agg / deg
        out = torch.cat([x, agg], dim=-1)
        return self.lin(out)


class SimpleGCNConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        row, col = edge_index
        deg = torch.bincount(row, minlength=x.size(0)).clamp(min=1).float()
        norm = (deg[row].pow(-0.5) * deg[col].pow(-0.5)).unsqueeze(1)
        agg = torch.zeros_like(x)
        agg.index_add_(0, row, x[col] * norm)
        return self.lin(agg)


class GraphEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, layers: int, dropout: float, encoder_type: str = "graphsage"):
        super().__init__()
        self.dropout = dropout
        self.encoder_type = encoder_type.lower()
        convs = []
        for layer_idx in range(layers):
            in_ch = in_dim if layer_idx == 0 else hidden_dim
            if HAS_PYG:
                if self.encoder_type == "gcn":
                    conv = GCNConv(in_ch, hidden_dim)
                else:
                    conv = SAGEConv(in_ch, hidden_dim)
            else:
                if self.encoder_type == "gcn":
                    conv = SimpleGCNConv(in_ch, hidden_dim)
                else:
                    conv = SimpleSAGEConv(in_ch, hidden_dim)
            convs.append(conv)
        self.convs = nn.ModuleList(convs)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class CoeffProjector(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, coeff_dim: int, dropout: float):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Linear(hidden_dim, coeff_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.trunk(x))


class SpatialEncoderProjector(nn.Module):
    def __init__(self, encoder: GraphEncoder, projector: CoeffProjector):
        super().__init__()
        self.encoder = encoder
        self.projector = projector

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, batch: Optional[torch.Tensor] = None) -> torch.Tensor:
        node_emb = self.encoder(x, edge_index)
        if batch is None:
            pooled = node_emb.mean(dim=0, keepdim=True)
        else:
            if HAS_PYG:
                pooled = global_mean_pool(node_emb, batch)
            else:
                pooled = node_emb.mean(dim=0, keepdim=True)
        return self.projector(pooled)


class LSTMTemporalHead(nn.Module):
    def __init__(self, coeff_dim: int, hidden_dim: int, layers: int, dropout: float, horizon: int):
        super().__init__()
        self.horizon = horizon
        self.lstm = nn.LSTM(
            input_size=coeff_dim,
            hidden_size=hidden_dim,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, horizon * coeff_dim)
        self.coeff_dim = coeff_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.lstm(x)
        last = hidden[-1]
        out = self.out(last)
        return out.view(x.size(0), self.horizon, self.coeff_dim)


class TransformerTemporalHead(nn.Module):
    def __init__(self, coeff_dim: int, hidden_dim: int, layers: int, dropout: float, horizon: int, n_heads: int, ff_dim: int):
        super().__init__()
        self.horizon = horizon
        self.coeff_dim = coeff_dim
        self.input_proj = nn.Linear(coeff_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.out = nn.Linear(hidden_dim, horizon * coeff_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.encoder(x)
        pooled = x[:, -1]
        out = self.out(pooled)
        return out.view(x.size(0), self.horizon, self.coeff_dim)


class MambaTemporalHead(nn.Module):
    def __init__(self, coeff_dim: int, hidden_dim: int, layers: int, dropout: float, horizon: int):
        super().__init__()
        self.horizon = horizon
        self.coeff_dim = coeff_dim
        if HAS_MAMBA:
            self.input_proj = nn.Linear(coeff_dim, hidden_dim)
            self.blocks = nn.ModuleList([Mamba(d_model=hidden_dim) for _ in range(layers)])
            self.out = nn.Linear(hidden_dim, horizon * coeff_dim)
        else:
            self.gru = nn.GRU(coeff_dim, hidden_dim, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
            self.out = nn.Linear(hidden_dim, horizon * coeff_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if HAS_MAMBA:
            h = self.input_proj(x)
            for block in self.blocks:
                h = block(h)
            pooled = h[:, -1]
        else:
            _, hidden = self.gru(x)
            pooled = hidden[-1]
        out = self.out(pooled)
        return out.view(x.size(0), self.horizon, self.coeff_dim)


def build_temporal_head(head: str, coeff_dim: int, cfg: Config) -> nn.Module:
    head = head.lower()
    if head == "lstm":
        return LSTMTemporalHead(coeff_dim, cfg.TEMPORAL_HIDDEN, cfg.TEMPORAL_LAYERS, cfg.TEMPORAL_DROPOUT, cfg.FORECAST_HORIZON)
    if head == "transformer":
        return TransformerTemporalHead(
            coeff_dim,
            cfg.TEMPORAL_HIDDEN,
            cfg.TEMPORAL_LAYERS,
            cfg.TEMPORAL_DROPOUT,
            cfg.FORECAST_HORIZON,
            cfg.TRANSFORMER_HEADS,
            cfg.TRANSFORMER_FF,
        )
    if head == "mamba":
        return MambaTemporalHead(coeff_dim, cfg.TEMPORAL_HIDDEN, cfg.TEMPORAL_LAYERS, cfg.TEMPORAL_DROPOUT, cfg.FORECAST_HORIZON)
    raise ValueError(f"Unknown head: {head}")

## 9 — Training utilities

In [ ]:
class AverageMeter:
    def __init__(self):
        self.total = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1) -> None:
        self.total += val * n
        self.count += n

    @property
    def avg(self) -> float:
        return self.total / max(self.count, 1)


def make_graph_data(edge_index: torch.Tensor) -> Any:
    if HAS_PYG:
        return Data(edge_index=edge_index)
    return {"edge_index": edge_index}


def set_graph_x(data: Any, x: torch.Tensor) -> None:
    if HAS_PYG:
        data.x = x
    else:
        data["x"] = x


def get_graph_x(data: Any) -> torch.Tensor:
    return data.x if HAS_PYG else data["x"]


def get_graph_edge_index(data: Any) -> torch.Tensor:
    return data.edge_index if HAS_PYG else data["edge_index"]


def build_node_features(t_idx: int, cfg: Config) -> np.ndarray:
    features = []
    if cfg.USE_NODE_COORDS:
        features.append(coords_norm)
    if cfg.USE_NODE_FIELDS:
        fields = np.stack(
            [
                norm_fields["pressure"][:, t_idx],
                norm_fields["u"][:, t_idx],
                norm_fields["v"][:, t_idx],
            ],
            axis=1,
        )
        features.append(fields)
    return np.concatenate(features, axis=1).astype(np.float32)


def build_graph_loader(data: Any, cfg: Config):
    if not HAS_PYG:
        return None
    if cfg.GRAPH_LOADER == "cluster":
        cluster_cache = CACHE_DIR / "cluster_data.pt"
        if cluster_cache.exists():
            cluster_data = torch.load(cluster_cache)
        else:
            base = Data(edge_index=data.edge_index, num_nodes=data.num_nodes)
            cluster_data = ClusterData(base, num_parts=cfg.CLUSTER_PARTS)
            torch.save(cluster_data, cluster_cache)
        cluster_data.data.x = data.x
        return ClusterLoader(cluster_data, batch_size=cfg.CLUSTER_BATCH_SIZE, shuffle=True)
    return NeighborLoader(
        data,
        num_neighbors=[cfg.K_NEIGHBORS] * cfg.ENCODER_LAYERS,
        batch_size=cfg.BATCH_SIZE,
        input_nodes=None,
        shuffle=True,
    )


def spatial_forward(model: SpatialEncoderProjector, data: Any, loader) -> torch.Tensor:
    if loader is None:
        x = get_graph_x(data).to(DEVICE)
        edge = get_graph_edge_index(data).to(DEVICE)
        return model(x, edge)

    coeff_sum = None
    total_nodes = 0.0
    for batch in loader:
        batch = batch.to(DEVICE)
        coeffs = model(batch.x, batch.edge_index, getattr(batch, "batch", None))
        weight = float(batch.num_nodes)
        coeff_sum = coeffs * weight if coeff_sum is None else coeff_sum + coeffs * weight
        total_nodes += weight
    return coeff_sum / max(total_nodes, 1.0)


def target_coeffs_at(t_idx: int, use_joint: bool) -> np.ndarray:
    if use_joint:
        return np.concatenate(
            [
                pod_artifacts["pressure"].coeffs[t_idx],
                pod_artifacts["u"].coeffs[t_idx],
                pod_artifacts["v"].coeffs[t_idx],
            ],
            axis=0,
        ).astype(np.float32)
    return pod_artifacts["pressure"].coeffs[t_idx].astype(np.float32)


def train_spatial_encoder(cfg: Config) -> SpatialEncoderProjector:
    in_dim = 0
    if cfg.USE_NODE_COORDS:
        in_dim += 2
    if cfg.USE_NODE_FIELDS:
        in_dim += 3

    coeff_dim = sum(art.coeffs.shape[1] for art in pod_artifacts.values())
    encoder = GraphEncoder(in_dim, cfg.ENCODER_HIDDEN, cfg.ENCODER_LAYERS, cfg.ENCODER_DROPOUT, cfg.ENCODER_TYPE)
    projector = CoeffProjector(cfg.ENCODER_HIDDEN, cfg.PROJ_HIDDEN, coeff_dim, cfg.PROJ_DROPOUT)
    model = SpatialEncoderProjector(encoder, projector).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scaler = GradScaler(enabled=cfg.USE_AMP)

    ckpt_path = CHECKPOINT_DIR / "spatial_encoder.pt"
    if cfg.RESUME and ckpt_path.exists():
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model"])
        print("Resumed spatial encoder from", ckpt_path)
        return model

    edge_t = torch.from_numpy(edge_index)
    data = make_graph_data(edge_t)

    def compute_loss(indices):
        model.eval()
        meter = AverageMeter()
        with torch.no_grad():
            for t_idx in indices:
                node_features = build_node_features(t_idx, cfg)
                set_graph_x(data, torch.from_numpy(node_features))
                loader = build_graph_loader(data, cfg)
                pred = spatial_forward(model, data, loader).squeeze(0)
                target = torch.from_numpy(target_coeffs_at(t_idx, True)).to(DEVICE)
                loss = F.mse_loss(pred, target)
                meter.update(loss.item())
        return meter.avg

    for epoch in range(cfg.SPATIAL_EPOCHS):
        model.train()
        meter = AverageMeter()
        for t_idx in range(0, train_end, cfg.SPATIAL_TIME_STRIDE):
            node_features = build_node_features(t_idx, cfg)
            set_graph_x(data, torch.from_numpy(node_features))
            loader = build_graph_loader(data, cfg)

            with autocast(enabled=cfg.USE_AMP):
                pred = spatial_forward(model, data, loader).squeeze(0)
                target = torch.from_numpy(target_coeffs_at(t_idx, True)).to(DEVICE)
                loss = F.mse_loss(pred, target)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            meter.update(loss.item())

        val_indices = range(train_end, val_end, cfg.SPATIAL_TIME_STRIDE)
        val_loss = compute_loss(val_indices) if val_end > train_end else np.inf
        print(f"[Spatial] Epoch {epoch+1}/{cfg.SPATIAL_EPOCHS} train={meter.avg:.4e} val={val_loss:.4e}")

    torch.save({"model": model.state_dict()}, ckpt_path)
    return model


def encode_coeffs_with_spatial_encoder(cfg: Config, model: SpatialEncoderProjector) -> np.ndarray:
    coeff_path = CACHE_DIR / "spatial_coeffs.npy"
    if coeff_path.exists():
        return np.load(coeff_path, mmap_mode="r+")

    coeff_dim = sum(art.coeffs.shape[1] for art in pod_artifacts.values())
    coeffs = np.lib.format.open_memmap(
        coeff_path,
        mode="w+",
        dtype=np.float32,
        shape=(n_timesteps, coeff_dim),
    )

    edge_t = torch.from_numpy(edge_index)
    data = make_graph_data(edge_t)

    model.eval()
    with torch.no_grad():
        for t_idx in range(n_timesteps):
            node_features = build_node_features(t_idx, cfg)
            set_graph_x(data, torch.from_numpy(node_features))
            loader = build_graph_loader(data, cfg)
            coeffs[t_idx] = spatial_forward(model, data, loader).squeeze(0).cpu().numpy()

    return coeffs


def train_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, scaler: GradScaler, cfg: Config) -> float:
    model.train()
    meter = AverageMeter()
    optimizer.zero_grad(set_to_none=True)

    for step, (x, y) in enumerate(loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with autocast(enabled=cfg.USE_AMP):
            pred = model(x)
            loss = F.mse_loss(pred, y)
            loss = loss / cfg.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % cfg.GRAD_ACCUM_STEPS == 0:
            if cfg.GRAD_CLIP is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        meter.update(loss.item() * cfg.GRAD_ACCUM_STEPS, n=x.size(0))

    return meter.avg


def eval_epoch(model: nn.Module, loader: DataLoader, cfg: Config) -> float:
    model.eval()
    meter = AverageMeter()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            pred = model(x)
            loss = F.mse_loss(pred, y)
            meter.update(loss.item(), n=x.size(0))
    return meter.avg


def train_temporal_head(
    head_name: str,
    coeff_dim: int,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: Config,
) -> Tuple[nn.Module, Dict[str, List[float]]]:
    model = build_temporal_head(head_name, coeff_dim, cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scaler = GradScaler(enabled=cfg.USE_AMP)

    best_val = float("inf")
    patience = 0
    history = {"train": [], "val": [], "epoch_time_sec": []}

    ckpt_path = CHECKPOINT_DIR / f"temporal_{head_name}.pt"
    if cfg.RESUME and ckpt_path.exists():
        checkpoint = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optim"])
        best_val = checkpoint.get("best_val", best_val)
        print(f"Resumed temporal head '{head_name}' from {ckpt_path}")

    for epoch in range(cfg.EPOCHS):
        epoch_start = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scaler, cfg)
        val_loss = eval_epoch(model, val_loader, cfg)
        epoch_time = time.time() - epoch_start
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        history["epoch_time_sec"].append(epoch_time)

        improved = val_loss < best_val
        if improved:
            best_val = val_loss
            patience = 0
            torch.save({"model": model.state_dict(), "optim": optimizer.state_dict(), "best_val": best_val}, ckpt_path)
        else:
            patience += 1

        print(f"[{head_name}] Epoch {epoch+1}/{cfg.EPOCHS} train={train_loss:.4e} val={val_loss:.4e}")
        if patience >= cfg.PATIENCE:
            print(f"Early stopping for {head_name} at epoch {epoch+1}")
            break

    return model, history

## 10 — Benchmark runner

In [ ]:
class MetricsAccumulator:
    def __init__(self, horizon: int):
        self.horizon = horizon
        self.sum_sq = np.zeros(horizon, dtype=np.float64)
        self.sum_abs = np.zeros(horizon, dtype=np.float64)
        self.sum_sq_true = np.zeros(horizon, dtype=np.float64)
        self.count = np.zeros(horizon, dtype=np.int64)

    def update(self, pred: np.ndarray, true: np.ndarray, basis: np.ndarray, mean: np.ndarray, scale: Dict[str, float], chunk_nodes: int = 50_000):
        horizon = pred.shape[1]
        for h in range(horizon):
            pred_h = pred[:, h]
            true_h = true[:, h]
            for n0 in range(0, basis.shape[1], chunk_nodes):
                n1 = min(basis.shape[1], n0 + chunk_nodes)
                basis_chunk = basis[:, n0:n1]
                pred_field = pred_h @ basis_chunk + mean[n0:n1]
                true_field = true_h @ basis_chunk + mean[n0:n1]
                pred_field = pred_field * scale["std"] + scale["mean"]
                true_field = true_field * scale["std"] + scale["mean"]

                diff = pred_field - true_field
                self.sum_sq[h] += np.square(diff).sum()
                self.sum_abs[h] += np.abs(diff).sum()
                self.sum_sq_true[h] += np.square(true_field).sum()
                self.count[h] += diff.size

    def finalize(self) -> Dict[str, np.ndarray]:
        rmse = np.sqrt(self.sum_sq / np.maximum(self.count, 1))
        mae = self.sum_abs / np.maximum(self.count, 1)
        rel_rmse = np.sqrt(self.sum_sq / np.maximum(self.sum_sq_true, 1e-12))
        return {"rmse": rmse, "mae": mae, "rel_rmse": rel_rmse}


def evaluate_temporal_head(
    model: nn.Module,
    loader: DataLoader,
    pod_artifacts: Dict[str, PODArtifacts],
    coeff_slices: Dict[str, slice],
    cfg: Config,
) -> Tuple[Dict[str, Dict[str, np.ndarray]], float]:
    model.eval()
    start_time = time.time()
    accumulators = {name: MetricsAccumulator(cfg.FORECAST_HORIZON) for name in coeff_slices.keys()}

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            pred = model(x).cpu().numpy()
            true = y.cpu().numpy()

            for name, slc in coeff_slices.items():
                pred_var = pred[:, :, slc]
                true_var = true[:, :, slc]
                art = pod_artifacts[name]
                scale = norm_stats[name]
                accumulators[name].update(pred_var, true_var, art.basis, art.mean, scale)

    metrics = {name: acc.finalize() for name, acc in accumulators.items()}
    inference_time = time.time() - start_time
    return metrics, inference_time


def run_benchmark(cfg: Config) -> Dict[str, Any]:
    results = {"history": {}, "metrics": {}, "timings": {}}

    if cfg.USE_JOINT_TRAINING:
        coeff_matrix_true, coeff_slices_local = build_coeff_matrix(pod_artifacts, True)
        input_coeffs = coeff_matrix_true
        if not cfg.USE_DIRECT_POD_COEFFS:
            print("Training spatial encoder for coefficient projection...")
            spatial_model = train_spatial_encoder(cfg)
            input_coeffs = encode_coeffs_with_spatial_encoder(cfg, spatial_model)

        train_loader, val_loader, test_loader = build_dataloaders(input_coeffs, coeff_matrix_true, cfg)

        for head in ("lstm", "transformer", "mamba"):
            start_time = time.time()
            model, history = train_temporal_head(head, input_coeffs.shape[1], train_loader, val_loader, cfg)
            train_time = time.time() - start_time

            metrics, infer_time = evaluate_temporal_head(model, test_loader, pod_artifacts, coeff_slices_local, cfg)
            results["history"][head] = history
            results["metrics"][head] = metrics
            results["timings"][head] = {"train_time_sec": train_time, "inference_time_sec": infer_time, "epoch_time_sec": float(np.mean(history["epoch_time_sec"]))}

        return results

    # Per-variable training
    if not cfg.USE_DIRECT_POD_COEFFS:
        print("Per-variable mode uses direct POD coefficients for stability.")

    for name in ("pressure", "u", "v"):
        coeff_matrix_true = np.asarray(pod_artifacts[name].coeffs, dtype=np.float32)
        coeff_slices_local = {name: slice(0, coeff_matrix_true.shape[1])}
        train_loader, val_loader, test_loader = build_dataloaders(coeff_matrix_true, coeff_matrix_true, cfg)

        for head in ("lstm", "transformer", "mamba"):
            start_time = time.time()
            model, history = train_temporal_head(head, coeff_matrix_true.shape[1], train_loader, val_loader, cfg)
            train_time = time.time() - start_time

            metrics, infer_time = evaluate_temporal_head(model, test_loader, {name: pod_artifacts[name]}, coeff_slices_local, cfg)
            results["history"][f"{head}_{name}"] = history
            results["metrics"][f"{head}_{name}"] = metrics
            results["timings"][f"{head}_{name}"] = {"train_time_sec": train_time, "inference_time_sec": infer_time, "epoch_time_sec": float(np.mean(history["epoch_time_sec"]))}

    return results


def summarize_results(results: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for head, metrics in results["metrics"].items():
        row = {"head": head}
        for var_name, vals in metrics.items():
            row[f"{var_name}_rmse"] = float(vals["rmse"].mean())
            row[f"{var_name}_rel_rmse"] = float(vals["rel_rmse"].mean())
        row.update(results["timings"].get(head, {}))
        rows.append(row)
    return pd.DataFrame(rows)

## 11 — Evaluation + plots

In [ ]:
def plot_training_curves(history: Dict[str, List[float]], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history["train"], label="train")
    ax.plot(history["val"], label="val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def plot_rmse_curves(metrics: Dict[str, Dict[str, np.ndarray]], out_path: Path) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, name in zip(axes, ("pressure", "u", "v")):
        if name not in metrics:
            continue
        ax.plot(metrics[name]["rmse"], label=f"{name} RMSE")
        ax.set_title(name)
        ax.set_xlabel("Horizon")
        ax.set_ylabel("RMSE")
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def reconstruct_field(
    coeffs: np.ndarray,
    art: PODArtifacts,
    scale: Dict[str, float],
) -> np.ndarray:
    field = coeffs @ art.basis + art.mean
    return field * scale["std"] + scale["mean"]


def plot_spatial_maps(
    coords: np.ndarray,
    true_field: np.ndarray,
    pred_field: np.ndarray,
    name: str,
    out_path: Path,
    max_cells: int,
) -> None:
    if coords.shape[0] > max_cells:
        idx = np.linspace(0, coords.shape[0] - 1, max_cells, dtype=int)
        coords = coords[idx]
        true_field = true_field[idx]
        pred_field = pred_field[idx]

    err = pred_field - true_field
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, data, title in zip(axes, (true_field, pred_field, err), ("GT", "Pred", "Error")):
        sc = ax.scatter(coords[:, 0], coords[:, 1], c=data, s=2, cmap="viridis")
        ax.set_title(f"{name} {title}")
        fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def plot_timeseries(
    true_series: np.ndarray,
    pred_series: np.ndarray,
    out_path: Path,
    name: str,
) -> None:
    fig, ax = plt.subplots(figsize=(6, 4))
    for i in range(true_series.shape[0]):
        ax.plot(true_series[i], label=f"GT {i}")
        ax.plot(pred_series[i], linestyle="--", label=f"Pred {i}")
    ax.set_title(f"Sampled cells: {name}")
    ax.set_xlabel("Timestep")
    ax.set_ylabel(name)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def visualise_samples(
    model: nn.Module,
    head: str,
    loader: DataLoader,
    coeff_slices: Dict[str, slice],
    cfg: Config,
) -> None:
    model.eval()
    x, y = next(iter(loader))
    x = x.to(DEVICE)
    with torch.no_grad():
        pred = model(x).cpu().numpy()
    true = y.numpy()

    for name, slc in coeff_slices.items():
        pred_coeffs = pred[0, :, slc]
        true_coeffs = true[0, :, slc]
        art = pod_artifacts[name]
        scale = norm_stats[name]

        true_field = reconstruct_field(true_coeffs, art, scale)
        pred_field = reconstruct_field(pred_coeffs, art, scale)

        max_timestep = true_field.shape[0] - 1
        valid_ts = [t for t in cfg.MAP_TIMESTEPS if t <= max_timestep]
        invalid_ts = [t for t in cfg.MAP_TIMESTEPS if t > max_timestep]
        if invalid_ts:
            print(f"Skipping timesteps {invalid_ts} for {name}. Max available: {max_timestep}")
        if not valid_ts:
            print(f"No valid map timesteps for {name}. Max available: {max_timestep}")
        for t in valid_ts:
            out_path = OUTPUT_DIR / f"{head}_{name}_map_t{t}.png"
            plot_spatial_maps(coords, true_field[t], pred_field[t], f"{name} t={t}", out_path, cfg.PLOT_MAX_CELLS)

        cell_ids = np.linspace(0, n_cells - 1, cfg.TIMESERIES_CELLS, dtype=int)
        true_series = true_field[:, cell_ids].T
        pred_series = pred_field[:, cell_ids].T
        ts_path = OUTPUT_DIR / f"{head}_{name}_timeseries.png"
        plot_timeseries(true_series, pred_series, ts_path, name)

## 12 — Summary + run recommendations

In [ ]:
results = run_benchmark(cfg)
summary_df = summarize_results(results)

summary_path = OUTPUT_DIR / "benchmark_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(summary_df)
print(f"Saved summary -> {summary_path}")

# Save training curves and RMSE curves
for head, history in results["history"].items():
    curve_path = OUTPUT_DIR / f"{head}_training_curve.png"
    plot_training_curves(history, curve_path)

    metrics = results["metrics"][head]
    rmse_path = OUTPUT_DIR / f"{head}_rmse_curve.png"
    plot_rmse_curves(metrics, rmse_path)

# Visualize one batch per head (skip if data too short)
if cfg.USE_JOINT_TRAINING:
    coeff_matrix_true, coeff_slices_local = build_coeff_matrix(pod_artifacts, True)
    input_coeffs = coeff_matrix_true
    if not cfg.USE_DIRECT_POD_COEFFS:
        input_coeffs = np.load(CACHE_DIR / "spatial_coeffs.npy", mmap_mode="r+")
    _, _, test_loader = build_dataloaders(input_coeffs, coeff_matrix_true, cfg)

    if len(test_loader) > 0:
        for head in ("lstm", "transformer", "mamba"):
            ckpt_path = CHECKPOINT_DIR / f"temporal_{head}.pt"
            if ckpt_path.exists():
                model = build_temporal_head(head, input_coeffs.shape[1], cfg).to(DEVICE)
                model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)["model"])
                visualise_samples(model, head, test_loader, coeff_slices_local, cfg)

print("Run complete.")
print("Recommended full run: set RUN_PROFILE='kaggle_full' and USE_MOCK_DATA=False.")